# Training Regression - Multicomponent

In [1]:
import os
import chemprop
from lightning import pytorch as pl
import torch
import numpy as np
import pandas as pd
from pathlib import Path

from chemprop import data, featurizers, models, nn
from chemprop.nn import metrics
from chemprop.models import multi

## Change your data inputs here

In [16]:
#input_path = r"C:\Users\Elwuz\OneDrive\Documents\GitHub\Code Projects\uvvisml\uvvisml\data\processed\all_lambda_max_abs_including_duplicates_and_song.csv" # path to your data .csv file containing SMILES strings and target values
input_path = r"C:\Users\Elwuz\OneDrive\Documents\GitHub\Code Projects\uvvisml\uvvisml\data\processed\all_optical_data_including_duplicates.csv"
smiles_columns = ['smiles', 'solvent'] # name of the column containing SMILES strings
target_columns = ['emission_max'] # list of names of the columns containing targets

df_input = pd.read_csv(input_path, usecols=['smiles', 'solvent', 'emission_max'])
df_input.dropna(inplace=True)

## Read data

In [17]:
df_input = pd.read_csv(input_path) #encoding="cp1252")
df_input

,smiles,solvent,absorption_max,emission_max,quantum_yield,log(e/L mol-1 cm-1),emi_lifetime(ns),abs FWHM (cm-1),emi FWHM (cm-1),abs FWHM (nm),emi FWHM (nm),source
0,COC(=O)c1ccn2cc(-c3ccc(OC)cc3)nc2c1,CS(C)=O,355.0,427.0,0.390,NaN,NaN,NaN,NaN,NaN,NaN,chemfluor
1,COC(=O)c1ccn2c(N)c(-c3ccc(OC)cc3)nc2c1,CS(C)=O,426.0,520.0,0.430,NaN,NaN,NaN,NaN,NaN,NaN,chemfluor
2,COC(=O)c1ccn2c(NC3CCCCC3)c(-c3ccc(OC)cc3)nc2c1,CS(C)=O,387.0,528.0,0.490,NaN,NaN,NaN,NaN,NaN,NaN,chemfluor
3,COC(=O)c1cccc2nc(-c3ccc(OC)cc3)c(NC3CCCCC3)n12,CS(C)=O,355.0,627.0,0.002,NaN,NaN,NaN,NaN,NaN,NaN,chemfluor
4,COc1ccc(-c2nc3ncccn3c2NC2CCCCC2)cc1,CS(C)=O,365.0,560.0,0.009,NaN,NaN,NaN,NaN,NaN,NaN,chemfluor
...,...,...,...,...,...,...,...,...,...,...,...,...
84678,S=C1CCCCC1,CCO,495.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,consolidation
84679,S=P1(c2ccccc2)C(c2ccccc2)=Cc2cc3ccccc3cc21,ClCCl,387.0,NaN,0.001,NaN,NaN,NaN,NaN,NaN,NaN,consolidation
84680,S=P1(c2ccccc2)c2cccc3c4c5ccccc5c5ccccc5c4c4ccc...,ClCCl,422.0,NaN,0.240,4.02,NaN,NaN,NaN,NaN,NaN,consolidation
84681,S=P1(c2ccccc2)c2cccc3ccc4ccc1n4c23,ClCCl,407.0,NaN,0.080,3.50,NaN,NaN,NaN,NaN,NaN,consolidation


## Get SMILES and targets

In [13]:
smiss = df_input.loc[:, smiles_columns].values
ys = df_input.loc[:, target_columns].values

In [14]:
# Take a look at the first 5 SMILES strings and targets
smiss[:5], ys[:5]

(array([['COC(=O)c1ccn2cc(-c3ccc(OC)cc3)nc2c1', 'CS(C)=O'],
        ['COC(=O)c1ccn2c(N)c(-c3ccc(OC)cc3)nc2c1', 'CS(C)=O'],
        ['COC(=O)c1ccn2c(NC3CCCCC3)c(-c3ccc(OC)cc3)nc2c1', 'CS(C)=O'],
        ['COC(=O)c1cccc2nc(-c3ccc(OC)cc3)c(NC3CCCCC3)n12', 'CS(C)=O'],
        ['COc1ccc(-c2nc3ncccn3c2NC2CCCCC2)cc1', 'CS(C)=O']], dtype=object),
 array([[427.],
        [520.],
        [528.],
        [627.],
        [560.]]))

## Make molecule datapoints
Create a list of lists containing the molecule datapoints for each components. The target is stored in the 0th component.

In [15]:
all_data = [[data.MoleculeDatapoint.from_smi(smis[0], y) for smis, y in zip(smiss, ys)]]
all_data += [[data.MoleculeDatapoint.from_smi(smis[i]) for smis in smiss] for i in range(1, len(smiles_columns))]

TypeError: No registered converter was able to produce a C++ rvalue of type class std::basic_string<wchar_t,struct std::char_traits<wchar_t>,class std::allocator<wchar_t> > from this Python object of type float

# Split data

## Perform data splitting for training, validation, and testing

In [67]:
component_to_split_by = 0 # index of the component to use for structure based splits
mols = [d.mol for d in all_data[component_to_split_by]]
train_indices, val_indices, test_indices = data.make_split_indices(mols, "scaffold_balanced", (0.8, 0.1, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    all_data, train_indices, val_indices, test_indices
)

The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
c:\Users\Elwuz\miniconda3\envs\chemprop-v2\Lib\site-packages\astartes\samplers\extrapolation\scaffold.py:48: NoMatchingScaffold: No matching scaffold was found for the 166 molecules corresponding to indices {26112, 26113, 26114, 77825, 77838, 72212, 72227, 72228, 72229, 62502, 62503, 72232, 10329, 10342, 12402, 12403, 12404, 12405, 12406, 12407, 12408, 12409, 12410, 12411, 12412, 12413, 12414, 12415, 12416, 12417, 12418, 12419, 10390, 10391, 79515, 62621, 10402, 79010, 76964, 81061, 11942, 11945, 76970, 72363, 11948, 11949, 11950, 11951, 11954, 11955, 11956, 11963, 10427, 11964, 10429, 10430, 11967, 11968, 11969, 11970, 11971, 62657, 62658, 62659, 62660, 11977, 62661, 62662, 62663, 62664, 62665, 62666, 62667, 62668, 62669, 11987, 62670, 11989, 12002, 12007, 12009, 12010, 12013, 12024, 12025, 12027, 12038, 12039, 61711, 60178, 12063, 12065, 12073, 12074, 12075, 79662, 12079, 12080, 12084, 12085, 694

In [68]:
#Or use preset spilts
#train_data = "uvvisml/data/splits/fluodb/scaffold/smiles_target_train.csv"
#val_data = "uvvisml/data/splits/fluodb/scaffold/smiles_target_val.csv" #Get these in proper form
#test_data = "uvvisml/data/splits/fluodb/scaffold/smiles_target_test.csv"

# Get MoleculeDataset for each components

In [69]:
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_datasets = [data.MoleculeDataset(train_data[0][i], featurizer) for i in range(len(smiles_columns))]
val_datasets = [data.MoleculeDataset(val_data[0][i], featurizer) for i in range(len(smiles_columns))]
test_datasets = [data.MoleculeDataset(test_data[0][i], featurizer) for i in range(len(smiles_columns))]

# Construct multicomponent dataset and scale the targets

In [70]:
train_mcdset = data.MulticomponentDataset(train_datasets)
scaler = train_mcdset.normalize_targets()
val_mcdset = data.MulticomponentDataset(val_datasets)
val_mcdset.normalize_targets(scaler)
test_mcdset = data.MulticomponentDataset(test_datasets)


# Construct data loader

In [71]:
BATCH_SIZE = 512
num_workers = 8 # number of workers for dataloader. 0 means using main process for data loading

train_loader = data.build_dataloader(train_mcdset, batch_size=BATCH_SIZE, num_workers=num_workers, pin_memory=True, persistent_workers=True)
val_loader = data.build_dataloader(val_mcdset, batch_size=BATCH_SIZE, num_workers=num_workers, pin_memory=True, persistent_workers=True, shuffle=False)
test_loader = data.build_dataloader(test_mcdset, batch_size=BATCH_SIZE, num_workers=num_workers, pin_memory=True, persistent_workers=True, shuffle=False)

# Construct multicomponent MPNN

## MulticomponentMessagePassing
- `blocks`: a list of message passing block used for each components
- `n_components`: number of components

In [72]:
mcmp = nn.MulticomponentMessagePassing(
    blocks=[nn.BondMessagePassing() for _ in range(len(smiles_columns))],
    n_components=len(smiles_columns),
)

## Aggregation

In [73]:
agg = nn.MeanAggregation()

## RegressionFFN

In [74]:
output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)

In [75]:
ffn = nn.RegressionFFN(
    input_dim=mcmp.output_dim,
    output_transform=output_transform,
)

## Metrics

In [76]:
metric_list = [metrics.RMSE(), metrics.MAE()] # Only the first metric is used for training and early stopping

## MulticomponentMPNN

In [77]:
mcmpnn = multi.MulticomponentMPNN(
    mcmp,
    agg,
    ffn,
    metrics=metric_list,
)

mcmpnn

MulticomponentMPNN(
  (message_passing): MulticomponentMessagePassing(
    (blocks): ModuleList(
      (0-1): 2 x BondMessagePassing(
        (W_i): Linear(in_features=86, out_features=300, bias=False)
        (W_h): Linear(in_features=300, out_features=300, bias=False)
        (W_o): Linear(in_features=372, out_features=300, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (tau): ReLU()
        (V_d_transform): Identity()
        (graph_transform): Identity()
      )
    )
  )
  (agg): MeanAggregation()
  (bn): Identity()
  (predictor): RegressionFFN(
    (ffn): MLP(
      (0): Sequential(
        (0): Linear(in_features=600, out_features=300, bias=True)
      )
      (1): Sequential(
        (0): ReLU()
        (1): Dropout(p=0.0, inplace=False)
        (2): Linear(in_features=300, out_features=1, bias=True)
      )
    )
    (criterion): MSE(task_weights=[[1.0]])
    (output_transform): UnscaleTransform()
  )
  (X_d_transform): Identity()
  (metrics): ModuleList(


# Set up trainer

In [ ]:
#code for training a model which will be saved
checkpoint = pl.callbacks.ModelCheckpoint(
    dirpath="checkpoints/",
    monitor="val_loss",
    save_top_k=1,
    mode="min",
    filename="song_model_10"
)

early_stop = pl.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    mode="min"
)

from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = torch.optim.AdamW(mcmpnn.parameters(), lr=7e-4, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

In [ ]:
trainer = pl.Trainer(
    logger=False,
    enable_checkpointing=True,
    enable_progress_bar=True,
    accelerator="gpu",
    devices=1,
    max_epochs=300, # number of epochs to train for
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


# Start training

In [80]:
trainer.fit(mcmpnn, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\Elwuz\miniconda3\envs\chemprop-v2\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory c:\Users\Elwuz\OneDrive\Documents\GitHub\Code Projects\uvvisml\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ MulticomponentMessagePassing │  455 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation              │      0 │ train │     0 │
│ 2 │ bn              │ Identity                     │      0 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN                │  180 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                     │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList                   │      0 │ train │     0 │
└───┴─────────────────┴──────────────────────────────┴────────┴───────┴───────┘

Trainable params: 636 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 636 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 35                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=500` reached.


# Test results

In [ ]:
results = trainer.test(mcmpnn, test_loader, weights_only=False)  # weights_only=False is only required with pytorch lightning version 2.6.0 or newer

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/mae          │    26.935359954833984     │
│         test/rmse         │     40.17124938964844     │
└───────────────────────────┴───────────────────────────┘

In [82]:
for cb in trainer.callbacks:
    if isinstance(cb, pl.callbacks.ModelCheckpoint):
        print(cb.best_model_score)  # validation score

None


In [83]:
import os
print(os.getcwd())

c:\Users\Elwuz\OneDrive\Documents\GitHub\Code Projects\uvvisml


In [84]:
import torch
from pathlib import Path
from chemprop.models import MPNN  # or MulticomponentMPNN if you used that

# pick your checkpoint
ckpt_path = "checkpoints/song_model_0.ckpt"

# load the model (weights and hyperparameters)
model = MPNN.load_from_checkpoint(ckpt_path, map_location="cpu")

# checkpoint itself stores metrics in `hparams` and `callbacks`
ckpt = torch.load(ckpt_path, map_location="cpu")

# see all keys
print(ckpt.keys())

# validation loss saved by Lightning
if 'callbacks' in ckpt:
    print(ckpt['callbacks'].keys())  # usually contains 'ModelCheckpoint'
    mc_ckpt = ckpt['callbacks']['ModelCheckpoint']
    print(mc_ckpt.keys())  # may include 'best_model_score', 'best_model_path'

# simplest:
print("Best val_loss:", ckpt['callbacks']['ModelCheckpoint']['best_model_score'])
print("Checkpoint path:", ckpt['callbacks']['ModelCheckpoint']['best_model_path'])

AttributeError: 'AttributeDict' object has no attribute 'hparams'